In [1]:
# 06_visualization.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 06 - Publication figures.
#
# All figures are grayscale (seaborn), carry no captions (captions live in the
# manuscript), and are written as BOTH png and pdf at 600 dpi into results/figures.
# Figures reflect the revised analysis: four techniques including the constrained-
# optimisation baseline, the paired technique-minus-baseline bootstrap intervals,
# the compliance-aggregator comparison, and the controlled signal-degradation study.
# No filesystem paths are printed (anonymity): only a count of written files is shown.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="white", context="paper")
plt.rcParams.update({"font.family": "serif", "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8,
                     "axes.grid": False, "savefig.bbox": "tight", "figure.dpi": 150})
DPI = 600
DATASET_ORDER = ["German Credit", "Taiwan CC Default", "Credit Card Approval"]
TECH_ORDER = ["baseline", "reweighing", "reject_option", "exp_gradient"]
TECH_LABELS = {"baseline": "Baseline", "reweighing": "Reweighing (pre)",
               "reject_option": "Reject option (post)", "exp_gradient": "Exp. gradient (in)"}
TECH_GRAY = {"baseline": "0.20", "reweighing": "0.45", "reject_option": "0.65", "exp_gradient": "0.82"}
_written = []


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=DPI)
        _written.append(f"{name}.{ext}")
    plt.close(fig)


def read_tab(name):
    return pd.read_csv(os.path.join(TAB_DIR, name))


rcfs_df = read_tab("04_rcfs_by_technique.csv")
sweep_df = read_tab("04_rcfs_gamma_sweep.csv")
agg_df = read_tab("04_aggregator_comparison.csv")
pbdiff = read_tab("05_paired_bootstrap_diff.csv")
adv_sweep = read_tab("07_adversarial_lambda_sweep.csv")
deg = read_tab("07_signal_degradation.csv")
techs_present = [t for t in TECH_ORDER if t in set(rcfs_df["technique"])]

# Figure 1: RCFS per dataset x technique (grouped bars).
piv = rcfs_df.pivot(index="dataset", columns="technique", values="RCFS").reindex(
    index=DATASET_ORDER, columns=techs_present)
fig, ax = plt.subplots(figsize=(7.0, 4.0))
x = np.arange(len(DATASET_ORDER)); width = 0.8 / len(techs_present)
for i, tech in enumerate(techs_present):
    ax.bar(x + (i - (len(techs_present) - 1) / 2) * width, piv[tech].values, width,
           label=TECH_LABELS[tech], color=TECH_GRAY[tech], edgecolor="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel("RCFS"); ax.set_ylim(0, max(0.8, np.nanmax(piv.values) * 1.15))
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
sns.despine(ax=ax); save_fig(fig, "fig1_rcfs_by_technique")

# Figure 2: compliance component scores per dataset (baseline technique).
base_rows = rcfs_df[rcfs_df["technique"] == "baseline"].set_index("dataset").reindex(DATASET_ORDER)
comp_cols = ["c_perf", "c_fair", "c_expl"]; comp_labels = ["Performance", "Fairness", "Explainability"]
comp_gray = ["0.20", "0.50", "0.78"]; comp_hatch = ["", "//", ".."]
fig, ax = plt.subplots(figsize=(6.4, 4.0)); x = np.arange(len(DATASET_ORDER)); width = 0.25
for i, (col, lab, g, h) in enumerate(zip(comp_cols, comp_labels, comp_gray, comp_hatch)):
    ax.bar(x + (i - 1) * width, base_rows[col].values, width, label=lab,
           color=g, edgecolor="black", linewidth=0.7, hatch=h)
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel("Component satisfaction score"); ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=9, loc="upper right")
sns.despine(ax=ax); save_fig(fig, "fig2_component_scores")

# Figure 3: RCFS vs gamma, one panel per dataset.
marker = {"baseline": "o", "reweighing": "s", "reject_option": "^", "exp_gradient": "D"}
ls = {"baseline": "-", "reweighing": "--", "reject_option": ":", "exp_gradient": "-."}
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharey=True)
for ax, ds in zip(axes, DATASET_ORDER):
    sub = sweep_df[sweep_df["dataset"] == ds]
    for tech in techs_present:
        s = sub[sub["technique"] == tech].sort_values("gamma")
        ax.plot(s["gamma"], s["RCFS"], marker=marker[tech], linestyle=ls[tech],
                color=TECH_GRAY[tech], markerfacecolor="white", markeredgecolor="black",
                linewidth=1.4, markersize=5, label=TECH_LABELS[tech])
    ax.set_title(ds, fontsize=10); ax.set_xlabel(r"$\gamma$"); sns.despine(ax=ax)
axes[0].set_ylabel("RCFS"); axes[-1].legend(frameon=False, fontsize=7, loc="best")
fig.tight_layout(); save_fig(fig, "fig3_gamma_curves")

# Figure 4: paired technique-minus-baseline RCFS with 95% bootstrap CI (zero line marked).
fig, ax = plt.subplots(figsize=(7.2, 4.2))
pb = pbdiff[pbdiff["technique"] != "baseline"].copy()
diff_techs = [t for t in ["reweighing", "reject_option", "exp_gradient"] if t in set(pb["technique"])]
x = np.arange(len(DATASET_ORDER)); width = 0.8 / len(diff_techs)
for i, tech in enumerate(diff_techs):
    sub = pb[pb["technique"] == tech].set_index("dataset").reindex(DATASET_ORDER)
    means = sub["mean_diff"].values
    lo = means - sub["CI_low"].values; hi = sub["CI_high"].values - means
    ax.bar(x + (i - (len(diff_techs) - 1) / 2) * width, means, width,
           label=TECH_LABELS[tech], color=TECH_GRAY[tech], edgecolor="black", linewidth=0.7,
           yerr=[lo, hi], capsize=3, error_kw={"elinewidth": 1.0, "ecolor": "black"})
ax.axhline(0.0, color="black", linewidth=0.9, linestyle="--")
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel(r"RCFS(technique) $-$ RCFS(baseline)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
sns.despine(ax=ax); save_fig(fig, "fig4_paired_diff_ci")

# Figure 5: adversarial RCFS vs adv_lambda, one line per dataset.
ds_style = {"German Credit": ("o", "-", "0.15"), "Taiwan CC Default": ("s", "--", "0.45"),
            "Credit Card Approval": ("^", ":", "0.70")}
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for ds in DATASET_ORDER:
    s = adv_sweep[adv_sweep["dataset"] == ds].sort_values("adv_lambda")
    mk, lstyle, g = ds_style[ds]
    ax.plot(s["adv_lambda"], s["RCFS"], marker=mk, linestyle=lstyle, color=g,
            markerfacecolor="white", markeredgecolor="black", linewidth=1.5, markersize=6, label=ds)
ax.set_xlabel(r"Adversarial strength $\lambda$"); ax.set_ylabel("RCFS")
ax.legend(frameon=False, fontsize=9); sns.despine(ax=ax); save_fig(fig, "fig5_adversarial_lambda")

# Figure 6: fairness and performance components along the lambda sweep, per dataset.
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharey=True)
for ax, ds in zip(axes, DATASET_ORDER):
    s = adv_sweep[adv_sweep["dataset"] == ds].sort_values("adv_lambda")
    ax.plot(s["adv_lambda"], s["c_fair"], marker="o", linestyle="-", color="0.15",
            markerfacecolor="white", markeredgecolor="black", linewidth=1.4, markersize=5, label="Fairness")
    ax.plot(s["adv_lambda"], s["c_perf"], marker="s", linestyle="--", color="0.45",
            markerfacecolor="white", markeredgecolor="black", linewidth=1.4, markersize=5, label="Performance")
    ax.plot(s["adv_lambda"], s["c_expl"], marker="^", linestyle=":", color="0.70",
            markerfacecolor="white", markeredgecolor="black", linewidth=1.4, markersize=5, label="Explainability")
    ax.set_title(ds, fontsize=10); ax.set_xlabel(r"$\lambda$"); sns.despine(ax=ax)
axes[0].set_ylabel("Component score"); axes[-1].legend(frameon=False, fontsize=7, loc="best")
fig.tight_layout(); save_fig(fig, "fig6_adversarial_tradeoff")

# Figure 7: compliance-aggregator comparison (RCFS under harmonic/geometric/arithmetic).
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), sharey=True)
agg_order = ["harmonic", "geometric", "arithmetic"]; agg_gray = {"harmonic": "0.20", "geometric": "0.50", "arithmetic": "0.78"}
for ax, ds in zip(axes, DATASET_ORDER):
    sub = agg_df[agg_df["dataset"] == ds]
    present = [t for t in TECH_ORDER if t in set(sub["technique"])]
    x = np.arange(len(present)); width = 0.25
    for i, ag in enumerate(agg_order):
        vals = [sub[(sub["technique"] == t) & (sub["aggregator"] == ag)]["RCFS"].values[0] for t in present]
        ax.bar(x + (i - 1) * width, vals, width, label=ag, color=agg_gray[ag],
               edgecolor="black", linewidth=0.6)
    ax.set_title(ds, fontsize=10); ax.set_xticks(x)
    ax.set_xticklabels([TECH_LABELS[t].split(" ")[0] for t in present], rotation=20, ha="right", fontsize=8)
    sns.despine(ax=ax)
axes[0].set_ylabel("RCFS"); axes[-1].legend(frameon=False, fontsize=8, loc="best", title=None)
fig.tight_layout(); save_fig(fig, "fig7_aggregator_comparison")

# Figure 8: controlled signal degradation on Taiwan - AUC and RCFS vs injected noise.
fig, ax1 = plt.subplots(figsize=(6.4, 4.0))
ax1.plot(deg["noise"], deg["AUC_baseline"], marker="o", color="0.15", linewidth=1.6,
         markerfacecolor="white", markeredgecolor="black", label="AUC (signal)")
ax1.set_xlabel("Injected label-noise rate"); ax1.set_ylabel("AUC"); ax1.set_ylim(0.5, 0.85)
ax2 = ax1.twinx()
ax2.plot(deg["noise"], deg["nn_baseline_RCFS"], marker="s", linestyle="--", color="0.45",
         markerfacecolor="white", markeredgecolor="black", linewidth=1.4, label="NN baseline RCFS")
ax2.plot(deg["noise"], deg["adversarial_RCFS"], marker="^", linestyle=":", color="0.70",
         markerfacecolor="white", markeredgecolor="black", linewidth=1.4, label="Adversarial RCFS")
ax2.set_ylabel("RCFS")
l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, la1 + la2, frameon=False, fontsize=8, loc="lower left")
sns.despine(ax=ax1, right=False); save_fig(fig, "fig8_signal_degradation")

print(f"Figures written (png+pdf pairs): {len(_written)} files")
print("Notebook 06 complete.")


Figures written (png+pdf pairs): 16 files
Notebook 06 complete.
